# 개별종목 조합C — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합C 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합C의 피처 값만 지정합니다.
import json

COMBINATION = 'C'
FEATURE_COLUMNS = (
    'ret_5',
    'overnight_gap',
    'intraday_return',
    'close_location',
    'bb_position',
    'rsi_14',
    'volume_z_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합C 피처: ('ret_5', 'overnight_gap', 'intraday_return', 'close_location', 'bb_position', 'rsi_14', 'volume_z_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4633,0.5012,-0.0379,0.3651,0.3734,0.0721,0.3827,0.1977,0.3013
1,2,balanced,980,20150123,20150421,0.3738,0.3978,-0.0241,0.3493,0.3544,0.0358,0.3587,0.2476,0.3132
2,3,balanced,1210,20151228,20160328,0.3659,0.3762,-0.0103,0.3609,0.3612,0.0439,0.3531,0.3024,0.3405
3,4,balanced,1439,20161202,20170228,0.4105,0.4617,-0.0513,0.3603,0.3634,0.0529,0.3760,0.2401,0.3199
4,5,balanced,1669,20171113,20180207,0.3893,0.3901,-0.0007,0.3754,0.3779,0.0681,0.3750,0.2818,0.3416
5,6,balanced,1899,20181024,20190118,0.4002,0.3725,0.0277,0.3987,0.3991,0.1001,0.4099,0.3765,0.3915
6,7,balanced,2129,20190930,20191224,0.4341,0.4781,-0.0441,0.3824,0.3836,0.0786,0.3842,0.2717,0.3489
7,8,balanced,2359,20200902,20201130,0.3918,0.3476,0.0442,0.3889,0.3903,0.0868,0.3833,0.3626,0.3806
8,9,balanced,2589,20210806,20211105,0.3643,0.3914,-0.0271,0.3512,0.3731,0.0474,0.3664,0.2014,0.2842
9,10,balanced,2818,20220714,20221012,0.3389,0.3454,-0.0066,0.3327,0.3423,0.0178,0.3500,0.2246,0.2882


,OOS 폴드 평균
accuracy,0.3920
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0049
macro_f1,0.3681
balanced_accuracy,0.3733
mcc,0.0627
pr_auc_macro_ovr,0.3752
down_recall,0.2749
core_harmonic_mean,0.3338


재실행 명령: python scripts/run_stock_model_experiment.py
